# E6 · B+ tree order sweep

Run this notebook from the repository root or `analysis/`. It reads committed Docker measurements
from `results/timings.csv` and exports the report's E6 tables and figures. The timing unit is
microseconds per insertion or query. The line is the median over timing batches and the shaded
band spans the middle half of the batches (Q1–Q3).

E6 uses the same six Spotify track ID sizes and hit/miss query batches as E2. Each build
replicate uses a fresh tree. The 1% tempo range uses the complete dataset and the E3 query seed.
The nine tested orders are 3, 4, 8, 16, 32, 64, 128, 256 and 512.

## Session provenance and order rankings

The default selects the latest complete run matching the current committed B+ implementation and
the latest complete original B+ run with the same machine, workload source, Python version and
hash seed. Git tree entries verify source identity. Set either session ID explicitly to reproduce a
specific run. A missing phase stays blank. Tables keep the exact commit and session IDs so a
timing result can be traced back to its raw batches.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "analysis/member_c_analysis.py").is_file())
sys.path.insert(0, str(repo))
from analysis import member_c_analysis as c

rows, machines = c.load_data(repo)
out = Path(os.environ.get("DS4300_ANALYSIS_DIR", repo / "analysis"))
figures = out / "figures_member_c"
tables = out / "tables_member_c"
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold",
                     "axes.grid": True, "grid.alpha": 0.2})

BASELINE_SESSION = None
CURRENT_SESSION = None

manifest, selected, summary, leaders = c.e6_summaries(
    rows, repo, baseline_id=BASELINE_SESSION, final_id=CURRENT_SESSION
)
manifest.to_csv(tables / "E6_sessions.csv", index=False)
summary.to_csv(tables / "E6_summary.csv", index=False)
leaders.to_csv(tables / "E6_best_orders.csv", index=False)
display(manifest)
display(selected)
if leaders.empty:
    print("No complete E6 session is available yet.")
else:
    display(leaders[leaders["n_keys"].eq(114000)].round(3))

sources = c.source_objects(repo, selected["code_version"],
                           (*c.E6_WORKLOAD_SOURCE, *c.BPLUS_SOURCE))
sources.to_csv(tables / "E6_source_objects.csv", index=False)


## E6.1 · Order trade-offs at 114,000 rows

Compare medians for each operation independently. `default_32_over_best` gives the cost of the
default order relative to the lowest median at that size. The two leading orders can be close
enough that their timing intervals overlap. Report those as close measurements and avoid
claiming that a single order wins for every workload or machine.

In [ ]:
full = summary[summary["n_keys"].eq(114000)]
phase_colors = {"Original B+": "#a75228", "Current B+": "#106d82"}
if not full.empty:
    fig, axes = plt.subplots(2, 2, figsize=(9.6, 6.3), layout="constrained")
    for ax, (operation, title) in zip(axes.flat, c.E6_OPS.items()):
        data = full[full["operation"].eq(operation)]
        for phase, curve in data.groupby("phase", sort=False):
            curve = curve.sort_values("order")
            color = phase_colors[phase]
            ax.plot(curve["order"], curve["median_ns"] / 1000,
                    marker="o", markersize=4, color=color, label=phase)
            ax.fill_between(curve["order"], curve["q1_ns"] / 1000,
                            curve["q3_ns"] / 1000, color=color, alpha=0.15)
        ax.set(xscale="log", xticks=c.ORDERS, xticklabels=c.ORDERS,
               xlabel="B+ tree order", ylabel="Median µs / operation", title=title)
        if operation == "range":
            ax.set_yscale("log")
    axes[0, 0].legend(frameon=False)
    fig.suptitle("E6 | B+ tree order at 114,000 Spotify rows", fontsize=15, weight="bold")
    fig.savefig(figures / "E6_order_full.png", dpi=180, bbox_inches="tight")
    plt.show()


## E6.2 · Scaling across the six track ID sizes

Fit OLS to log elapsed time against log inserted rows for each order and operation. Build
uses total elapsed time because the question is how the whole build scales. Point operations
use elapsed time per query. These are effective slopes over the measured interval. A small
positive point slope is compatible with logarithmic lookup work and changing cache behavior.

In [ ]:
growth = c.e6_growth(summary)
growth.to_csv(tables / "E6_growth.csv", index=False)
display(growth.round(3))

if not full.empty:
    phase = "Current B+" if "Current B+" in summary["phase"].values else "Original B+"
    scale = summary[summary["phase"].eq(phase) & summary["key_column"].eq("track_id")]
    fig, axes = plt.subplots(1, 3, figsize=(12.6, 4.1), layout="constrained")
    colors = plt.get_cmap("tab10")(range(len(c.ORDERS)))
    for ax, (operation, title) in zip(axes, list(c.E6_OPS.items())[:3]):
        for color, (order, data) in zip(colors, scale[scale["operation"].eq(operation)].groupby("order")):
            data = data.sort_values("n_keys")
            factor = data["n_keys"] / 1_000_000 if operation == "build" else 1 / 1000
            ax.plot(data["n_keys"], data["median_ns"] * factor, marker=".",
                    color=color, label=str(order))
        unit = "Median ms / build" if operation == "build" else "Median µs / query"
        ax.set(xscale="log", yscale="log", xlabel="Track ID rows", ylabel=unit, title=title)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Order", ncol=9, frameon=False, loc="outside lower center")
    fig.suptitle(f"E6 | Growth by input size ({phase})", fontsize=15, weight="bold")
    fig.savefig(figures / "E6_track_id_scaling.png", dpi=180, bbox_inches="tight")
    plt.show()


## E6.3 · Original vs. current source

The paired table uses equal order, input and operation settings on the same machine.
Use range comparisons and visited nodes to explain the reduction in work. Height and splits
remain available in the E6 summary to distinguish structural effects from query traversal.

In [ ]:
paired = c.e6_change(summary)
paired.to_csv(tables / "E6_before_after.csv", index=False)
display(paired[paired["operation"].eq("range")].round(3))

if not full.empty:
    fig, axes = plt.subplots(2, 2, figsize=(9.6, 6.3), layout="constrained")
    panels = [
        ("build", "height", "Tree height", "Levels"),
        ("build", "splits_per_op", "Build splits", "Splits / insertion"),
        ("range", "nodes_visited_per_op", "Range node visits", "Nodes / query"),
        ("range", "comparisons_per_op", "Range key comparisons", "Comparisons / query"),
    ]
    for ax, (operation, metric, title, unit) in zip(axes.flat, panels):
        data = full[full["operation"].eq(operation)]
        for phase, curve in data.groupby("phase", sort=False):
            curve = curve.sort_values("order")
            ax.plot(curve["order"], curve[metric], marker="o", markersize=4,
                    color=phase_colors[phase], label=phase)
        ax.set(xscale="log", xticks=c.ORDERS, xticklabels=c.ORDERS,
               xlabel="B+ tree order", ylabel=unit, title=title)
        if metric != "height":
            ax.set_yscale("log")
    axes[0, 0].legend(frameon=False)
    if full["phase"].nunique() == 2:
        for ax, (_, metric, _, _) in zip(axes[0], panels[:2]):
            curves = full[full["operation"].eq("build")].pivot(
                index="order", columns="phase", values=metric
            )
            if curves["Original B+"].equals(curves["Current B+"]):
                ax.text(0.98, 0.52, "Curves overlap", transform=ax.transAxes,
                        ha="right", color="#555555")
    fig.suptitle("E6 | B+ tree structure and range work", fontsize=15, weight="bold")
    fig.savefig(figures / "E6_range_work.png", dpi=180, bbox_inches="tight")
    plt.show()
